In [1]:
import requests
import pandas as pd
import json
import os
import numpy as np
from time import sleep
from scipy.stats import norm
import warnings

# Configurações de exibição e avisos
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
warnings.filterwarnings('ignore')


In [2]:
ativos = ['ABEV3', 'B3SA3', 'BBAS3', 'BBSE3', 'EGIE3', 'FLRY3',
          'HYPE3', 'ITSA4', 'KLBN11', 'LEVE3', 'PETR4', 'TAEE11',
          'UNIP6', 'VALE3', 'RADL3', 'CMIG4']

In [ ]:
# ---------------------------------------------------
# 2. DOWNLOAD E ATUALIZAÇÃO (CORREÇÃO JSON)
# ---------------------------------------------------
if not os.path.exists("dbJson"):
    os.makedirs("dbJson")

def atualizarDados(ativo):
    url = f"https://storage.googleapis.com/api-cdn-eaglesystem/api/{ativo.upper()}"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return f"{ativo}: erro HTTP {response.status_code}"
        
        # Validação de conteúdo JSON para evitar erro 'Expecting value'
        content = response.text.strip()
        if not (content.startswith('{') or content.startswith('[')):
            return f"{ativo}: Erro - Conteúdo não é JSON válido"

        dados = response.json()
        with open(f"dbJson/{ativo}.json", "w", encoding="utf-8") as arq:
            json.dump(dados, arq, indent=2)
        return f"{ativo}: atualizado"
    except Exception as e:
        return f"{ativo}: erro -> {e}"

for ativo in ativos:
    print(atualizarDados(ativo))


ABEV3: atualizado
B3SA3: atualizado


In [ ]:
# ---------------------------------------------------
# 3. PROCESSAMENTO DOS DADOS (DATAFRAME)
# ---------------------------------------------------
def dataFrameUnico(ativo):
    with open(f"dbJson/{ativo}.json", "r", encoding="utf-8") as arq:
        dados = json.load(arq)

    preco_atual = dados["asset"]["close"]
    linhas = []

    for serie in dados["series"]:
        vencimento = serie.get("due_date")
        dias = serie.get("days_to_maturity")

        for strike in serie["strikes"]:
            strike_price = strike["strike"]
            for tipo in ["call", "put"]:
                opt = strike[tipo]
                if opt:
                    linhas.append({
                        "ativo": ativo, 
                        "symbol": opt["symbol"],
                        "categoria": tipo.upper(), 
                        "preco_atual": preco_atual,
                        "strike": strike_price, 
                        "bid": opt["bid"], 
                        "ask": opt["ask"], 
                        "volume": opt["volume"],
                        "maturity_type": opt["maturity_type"],
                        "moneyness": opt["bs"]["moneyness"],
                        "price": opt["bs"]["price"],
                        "delta": opt["bs"]["delta"], 
                        "gamma": opt["bs"]["gamma"], 
                        "vega": opt["bs"]["vega"],
                        "theta": opt["bs"]["theta"],
                        "rho": opt["bs"]["rho"],
                        "vol": opt["bs"]["volatility"], 
                        "poe": opt["bs"]["poe"],
                        "dias": dias,
                        "vencimento": vencimento,
                    })
    return pd.DataFrame(linhas)

todos = []
for ativo in ativos:

    df = dataFrameUnico(ativo)
    todos.append(df)


df_final = pd.concat(todos, ignore_index=True)

In [ ]:
df_final

In [ ]:
puts = df_final[df_final["categoria"] == "PUT"].copy()


puts["retorno"] = (puts["bid"] / (puts["strike"] - puts['bid'])) * 100
puts["retorno_mes"] = puts["retorno"] * (30 / puts["dias"].replace(0, 1))
puts["dist_strike"] = ((puts["strike"] / puts["preco_atual"]) - 1) * 100


# Ranking e Score
filtro_put = puts[

    (puts['dias'].between(15, np.inf)) &
    (puts['dist_strike'] <= 0) &
    (puts['retorno_mes'] >= 0.95)


].copy()

if not filtro_put.empty:
    filtro_put['score'] = (
        filtro_put['dist_strike'].rank(ascending=True) * 1 +
        filtro_put['retorno_mes'].rank(ascending=False) * 4 +
        filtro_put['retorno'].rank(ascending=False) * 3
    )
    print("\n--- MELHORES OPÇÕES DE VENDA DE PUT ---")

    display(filtro_put[

        [
            'ativo', 'symbol', 'categoria','moneyness' , 'strike', 'preco_atual',
            'dist_strike', 'bid', 'ask',
            'volume', 'delta', 'theta', 'vol', 'poe',  'retorno', 'retorno_mes',
            'dias', 'vencimento', 'score',
        ]

    ].sort_values('score').head(15))

In [ ]:
calls = df_final[df_final["categoria"] == "CALL"].copy()

# Retorno apenas do prêmio
calls["retorno"] = (calls["bid"] / calls["preco_atual"]) * 100

# Retorno mensalizado
calls["retorno_mes"] = calls["retorno"] * (30 / calls["dias"].replace(0, 1))

# Distância do strike
calls["dist_strike"] = (
    (calls["strike"] / calls["preco_atual"]) - 1
) * 100


# -----------------------------
# FILTRO
# -----------------------------

filtro_call = calls[

    (calls["dias"] >= 15) &
    (calls["dist_strike"].between(0,np.inf)) &
    (calls["retorno_mes"] >= 0.70)


].copy()

# -----------------------------
# SCORE
# -----------------------------

if not filtro_call.empty:

    filtro_call["score"] = (

        filtro_call["retorno_mes"].rank(ascending=False) * 4 +
        filtro_call["dist_strike"].rank(ascending=False) * 3 +
        filtro_call["volume"].rank(ascending=False) * 2 +
        filtro_call["theta"].rank(ascending=False) * 1

    )

    print("\n--- MELHORES OPÇÕES DE VENDA DE CALL ---")

    display(

        filtro_call[
            [
                "ativo", "symbol", "categoria", "moneyness", "strike",
                "preco_atual", "dist_strike", "bid", "ask", "volume", "delta",
                "theta", "vol", "poe", "retorno", "retorno_mes", "dias", "vencimento",
                "score",

            ]
        ]
        .sort_values("score")
        .head(30)

    )

In [ ]:
calls = df_final[
    (df_final["categoria"] == "CALL") &
    (df_final["maturity_type"] == "AMERICAN")
].copy()


# ==========================================================
# MÉTRICAS
# ==========================================================

# Distância do strike em relação ao preço atual
calls["dist_strike"] = ((calls["strike"] / calls["preco_atual"]) - 1) * 100

# Spread
calls["spread"] = calls["ask"] - calls["bid"]

# Spread percentual
calls["spread_pct"] = (calls["spread"] / calls["ask"].replace(0, np.nan)) * 100

# Retorno potencial (%)
# Quanto a ação precisa subir para atingir o strike em relação ao prêmio pago
calls["retorno"] = (
    ((calls["strike"] - calls["preco_atual"]) / calls["ask"])) * 100

# Retorno potencial mensalizado
calls["retorno_mes"] = (calls["retorno"] * (30 / calls["dias"].replace(0, 1)))

# ==========================================================
# FILTRO
# ==========================================================

filtro_call = calls[

    (calls["dias"] >= 90) &
    (calls["price"].between(0.01, 2)) &
    (calls["dist_strike"].between(-10, -1)) 
    


].copy()


# ==========================================================
# SCORE
# ==========================================================

if not filtro_call.empty:

    filtro_call["score"] = (

        filtro_call["price"].rank(ascending=True) * 4 +
        filtro_call["dias"].rank(ascending=False) * 2


    )

    print("\n--- MELHORES OPÇÕES PARA COMPRA DE CALL ---")

    display(

        filtro_call[
            [
                "ativo", "symbol", "categoria", "maturity_type", "moneyness", "strike", "preco_atual",
                "dist_strike", "price", "volume", "delta", "theta", "vol", "poe", "dias", "vencimento", "score",
            ]
        ]
        .sort_values("score")
        .head(30)

    )

In [ ]:
# Estudo pessoal

In [ ]:
estudo = df_final[
    (df_final["categoria"] == "CALL") &
    (df_final["maturity_type"] == "AMERICAN")
].copy()

filtro_estudo = estudo[
    (estudo["moneyness"] == 'ITM') &
    (estudo['price'] >= (estudo['preco_atual']-estudo['strike'])) &
    (estudo['price'] <= 1.5) &
    (estudo['dias'] >= 30)

]

filtro_estudo["score"] = (

    filtro_estudo["delta"].rank(ascending=True) * 4 +
    filtro_estudo["price"].rank(ascending=True) * 2


)

display(

    filtro_estudo[
        [
            "ativo", "symbol", "categoria", "maturity_type", "moneyness", "strike", "preco_atual",
            "price", "delta", "theta", "volume", "dias", "vencimento", "score"
        ]
    ]
    .sort_values("score")
    .head(30)

)